In [1]:
from pathlib import Path
import json

artifact_dir = Path("artifacts/agent_rounds/20260615_round4/artifact_lifecycle")
artifact_dir.mkdir(parents=True, exist_ok=True)

seed_files = {
    "keep.txt": "keep artifact created by the seed cell\n",
    "mutable.txt": "version=1\nsource=seed\n",
    "delete_me.txt": "created by seed; repaired cell should delete this\n",
}

for name, text in seed_files.items():
    (artifact_dir / name).write_text(text, encoding="utf-8")

(artifact_dir / "metrics.json").write_text(
    json.dumps(
        {
            "cell": "seed",
            "created": sorted(seed_files),
            "purpose": "artifact lifecycle baseline",
        },
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

print("seeded artifacts:")
for path in sorted(p for p in artifact_dir.rglob("*") if p.is_file()):
    print(f"- {path.as_posix()} ({path.stat().st_size} bytes)")


seeded artifacts:
- artifacts/agent_rounds/20260615_round4/artifact_lifecycle/delete_me.txt (50 bytes)
- artifacts/agent_rounds/20260615_round4/artifact_lifecycle/failure_leftover.txt (62 bytes)
- artifacts/agent_rounds/20260615_round4/artifact_lifecycle/keep.txt (39 bytes)
- artifacts/agent_rounds/20260615_round4/artifact_lifecycle/metrics.json (140 bytes)
- artifacts/agent_rounds/20260615_round4/artifact_lifecycle/mutable.txt (22 bytes)
- artifacts/agent_rounds/20260615_round4/artifact_lifecycle/nested/from_failed_cell/stale.txt (55 bytes)
- artifacts/agent_rounds/20260615_round4/artifact_lifecycle/nested/repaired/final.txt (45 bytes)
- artifacts/agent_rounds/20260615_round4/artifact_lifecycle/repair_summary.json (436 bytes)


In [2]:
from pathlib import Path
import json

artifact_dir = Path("artifacts/agent_rounds/20260615_round4/artifact_lifecycle")
artifact_dir.mkdir(parents=True, exist_ok=True)

mutable = artifact_dir / "mutable.txt"
previous_text = mutable.read_text(encoding="utf-8") if mutable.exists() else ""
mutable.write_text(
    "version=2\nsource=repaired mutation cell\n"
    f"previous_started_with_version_1={previous_text.startswith('version=1')}\n",
    encoding="utf-8",
)

delete_target = artifact_dir / "delete_me.txt"
deleted = delete_target.exists()
if deleted:
    delete_target.unlink()

nested = artifact_dir / "nested" / "repaired" / "final.txt"
nested.parent.mkdir(parents=True, exist_ok=True)
nested.write_text(
    "nested artifact created by the repaired cell\n",
    encoding="utf-8",
)

summary = {
    "cell": "repaired_mutate_delete_nested",
    "modified": mutable.as_posix(),
    "deleted_target": delete_target.as_posix(),
    "deleted_target_existed": deleted,
    "created_nested": nested.as_posix(),
    "failure_leftover_exists": (artifact_dir / "failure_leftover.txt").exists(),
    "failed_nested_exists": (
        artifact_dir / "nested" / "from_failed_cell" / "stale.txt"
    ).exists(),
}
(artifact_dir / "repair_summary.json").write_text(
    json.dumps(summary, indent=2) + "\n",
    encoding="utf-8",
)

print("repair actions:")
print(f"- modified: {mutable.as_posix()} ({mutable.stat().st_size} bytes)")
print(f"- deleted: {delete_target.as_posix()} existed={deleted}")
print(f"- created nested: {nested.as_posix()} ({nested.stat().st_size} bytes)")
print(f"- failure_leftover_exists={summary['failure_leftover_exists']}")
print(f"- failed_nested_exists={summary['failed_nested_exists']}")
print("current artifact files:")
for path in sorted(p for p in artifact_dir.rglob("*") if p.is_file()):
    print(f"- {path.relative_to(artifact_dir).as_posix()}")


repair actions:
- modified: artifacts/agent_rounds/20260615_round4/artifact_lifecycle/mutable.txt (77 bytes)
- deleted: artifacts/agent_rounds/20260615_round4/artifact_lifecycle/delete_me.txt existed=True
- created nested: artifacts/agent_rounds/20260615_round4/artifact_lifecycle/nested/repaired/final.txt (45 bytes)
- failure_leftover_exists=True
- failed_nested_exists=True
current artifact files:
- failure_leftover.txt
- keep.txt
- metrics.json
- mutable.txt
- nested/from_failed_cell/stale.txt
- nested/repaired/final.txt
- repair_summary.json


In [3]:
from pathlib import Path

artifact_dir = Path("artifacts/agent_rounds/20260615_round4/artifact_lifecycle")
paths = sorted(p for p in artifact_dir.rglob("*") if p.is_file())

print("artifact inventory:")
for path in paths:
    print(f"- {path.relative_to(artifact_dir).as_posix()} ({path.stat().st_size} bytes)")

print(f"delete_me_exists={(artifact_dir / 'delete_me.txt').exists()}")
print(f"failure_leftover_exists={(artifact_dir / 'failure_leftover.txt').exists()}")
print(
    "failed_nested_exists="
    f"{(artifact_dir / 'nested' / 'from_failed_cell' / 'stale.txt').exists()}"
)
print(f"repaired_nested_exists={(artifact_dir / 'nested' / 'repaired' / 'final.txt').exists()}")


artifact inventory:
- failure_leftover.txt (62 bytes)
- keep.txt (39 bytes)
- metrics.json (140 bytes)
- mutable.txt (77 bytes)
- nested/from_failed_cell/stale.txt (55 bytes)
- nested/repaired/final.txt (45 bytes)
- repair_summary.json (436 bytes)
delete_me_exists=False
failure_leftover_exists=True
failed_nested_exists=True
repaired_nested_exists=True
